In [2]:
# ============================================================
# Notebook 1: 01_preprocessing_grouping_model.ipynb
# Project : Double Exclusion — Intersectional Fairness in VIP Recourse
# Purpose : Load synthetic card data, define intersectional groups,
#           train the VIP classifier, and checkpoint all artifacts
#           needed by Notebook 2 (recourse generation).
# ============================================================

import os, glob, zipfile, pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, recall_score, roc_auc_score, classification_report

pd.set_option("display.max_columns", 100)

# ------------------------------------------------------------
# 0. Paths (edit DATA_DIR on your machine)
# ------------------------------------------------------------
DATA_DIR = r"C:\Users\miy\Downloads\1000_논문투고\117.금융 합성데이터\3.개방데이터\1.데이터\1. 합성데이터"
WORK_DIR = os.path.join(DATA_DIR, "_extracted")    # scratch + checkpoints
ARTIFACT_DIR = os.path.join(DATA_DIR, "_artifacts")
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(ARTIFACT_DIR, exist_ok=True)
assert os.path.isdir(DATA_DIR), "Source directory not found — check DATA_DIR."
ENC = "utf-8-sig"   # source CSVs are UTF-8 (BOM), comma-separated

# ------------------------------------------------------------
# 1. Unzip only the three tables this study needs
#    (member / approval / credit). CB sheets are excluded to
#    preserve the 'internal behavioral data only' assumption.
# ------------------------------------------------------------
TARGET_ZIPS = ["01", "02", "03"]
for zpath in glob.glob(os.path.join(DATA_DIR, "*.zip")):
    fname = os.path.basename(zpath)
    if any(t in fname[:4] for t in TARGET_ZIPS):
        with zipfile.ZipFile(zpath) as z:
            z.extractall(WORK_DIR)
        print("Extracted:", fname)

# ------------------------------------------------------------
# 2. Classify each CSV by its (Korean) header signature
# ------------------------------------------------------------
csv_paths = sorted(glob.glob(os.path.join(WORK_DIR, "*.csv")))
SIGNATURES = {
    "member":   "VIP등급코드",
    "approval": "이용금액_신용_R3M",
    "credit":   "카드이용한도금액",
}
def classify(path):
    cols = pd.read_csv(path, nrows=0, encoding=ENC).columns
    for label, sig in SIGNATURES.items():
        if sig in cols:
            return label
    return "unknown"

file_index = {}
for p in csv_paths:
    file_index.setdefault(classify(p), []).append(p)
for label, paths in sorted(file_index.items()):
    print(f"{label:10s}: {len(paths)} files")

# ------------------------------------------------------------
# 3. Select & rename core variables, load latest snapshot (201812)
# ------------------------------------------------------------
MEMBER_COLS = {
    "기준년월": "job_mon", "발급회원번호": "member_no",
    "남녀구분코드": "code_gender", "연령": "age",
    "VIP등급코드": "code_vip", "거주시도명": "add_home_si",
    "입회경과개월수_신용": "mn_credit_tenure", "Life_Stage": "code_life",
    "회원여부_연체": "yn_delinquent",
}
APPROVAL_COLS = {
    "발급회원번호": "member_no",
    "이용금액_신용_R3M": "amt_credit_r3m", "이용금액_업종기준": "amt_usage_r3m",
    "이용가맹점수": "cnt_merchant_r3m", "이용금액_쇼핑": "amt_shop",
    "이용금액_요식": "amt_eat", "이용금액_여유생활": "amt_leisure",
}
CREDIT_COLS = {"발급회원번호": "member_no", "카드이용한도금액": "amt_credit_limit"}

def load_latest(label, colmap):
    latest = max(file_index[label], key=lambda p: os.path.basename(p)[:6])
    return pd.read_csv(latest, encoding=ENC, usecols=list(colmap.keys())).rename(columns=colmap)

member   = load_latest("member",   MEMBER_COLS)
approval = load_latest("approval", APPROVAL_COLS)
credit   = load_latest("credit",   CREDIT_COLS)
print("Loaded:", member.shape, approval.shape, credit.shape)

df = (member
      .merge(approval.drop(columns=["job_mon"], errors="ignore"), on="member_no", how="inner")
      .merge(credit.drop(columns=["job_mon"], errors="ignore"),   on="member_no", how="inner"))
print("Merged:", df.shape, "| duplicate member_no:", df["member_no"].duplicated().sum())

# ------------------------------------------------------------
# 4. Binarize target (VIP vs. general)
# ------------------------------------------------------------
df["code_vip"] = df["code_vip"].astype(str).str.strip()
df["VIP"] = (~df["code_vip"].isin(["_", "nan", ""])).astype(int)
print("VIP rate:", round(df["VIP"].mean(), 4))

# ------------------------------------------------------------
# 5. Define intersectional groups (data-driven tenure cutoffs)
# ------------------------------------------------------------
TENURE_SHORT = df["mn_credit_tenure"].quantile(0.25)
TENURE_LONG  = df["mn_credit_tenure"].quantile(0.75)
MALE, FEMALE = 1, 2
YOUNG_BANDS, OLD_BANDS = ["30대"], ["60대", "70대이상"]

def assign_group(row):
    long_h  = row["mn_credit_tenure"] >= TENURE_LONG
    short_h = row["mn_credit_tenure"] <= TENURE_SHORT
    if row["age"] in YOUNG_BANDS and row["code_gender"] == MALE   and long_h:
        return "advantaged"
    if row["age"] in OLD_BANDS   and row["code_gender"] == FEMALE and short_h:
        return "disadvantaged"
    return "other"

df["group"] = df.apply(assign_group, axis=1)
print(f"Cutoffs: short<={TENURE_SHORT:.0f}m, long>={TENURE_LONG:.0f}m")
print(df.groupby("group")["VIP"].agg(["size", "mean"]))

# ------------------------------------------------------------
# 6. Feature governance (drives BOTH model features AND DiCE penalties)
# ------------------------------------------------------------
IMMUTABLE        = ["code_gender", "age", "add_home_si", "mn_credit_tenure", "code_life"]
ACTIONABLE       = ["amt_credit_r3m", "amt_usage_r3m", "cnt_merchant_r3m",
                    "amt_shop", "amt_eat", "amt_leisure"]
NON_CONTROLLABLE = ["amt_credit_limit"]
FEATURES = IMMUTABLE + ACTIONABLE + NON_CONTROLLABLE
TARGET   = "VIP"

model_df = df[FEATURES + [TARGET, "group"]].copy()
AGE_ORDER = ["20대", "30대", "40대", "50대", "60대", "70대이상"]
model_df["age"] = pd.Categorical(model_df["age"], categories=AGE_ORDER, ordered=True).codes
model_df["code_gender"] = model_df["code_gender"].astype("category").cat.codes
model_df["add_home_si"] = model_df["add_home_si"].astype("category").cat.codes
model_df["code_life"]   = model_df["code_life"].astype(str).astype("category").cat.codes

# ------------------------------------------------------------
# 7. Stratified large sample (preserve VIP ratio AND group mix)
# ------------------------------------------------------------
SAMPLE_N = 200000
strat_key = model_df["group"].astype(str) + "_" + model_df[TARGET].astype(str)
sample_df, _ = train_test_split(model_df, train_size=SAMPLE_N,
                                stratify=strat_key, random_state=42)
print("Sample VIP rate:", round(sample_df[TARGET].mean(), 4))
print(sample_df["group"].value_counts())

X = sample_df[FEATURES]; y = sample_df[TARGET]
strat2 = sample_df["group"].astype(str) + "_" + y.astype(str)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=strat2, random_state=42)
group_test = sample_df.loc[X_test.index, "group"]

# ------------------------------------------------------------
# 8. Train Random Forest (KCI-consistent, class-balanced)
# ------------------------------------------------------------
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                            random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]
print("F1     :", round(f1_score(y_test, y_pred), 4))
print("Recall :", round(recall_score(y_test, y_pred), 4))
print("AUC-ROC:", round(roc_auc_score(y_test, y_proba), 4))
print(classification_report(y_test, y_pred, digits=4))

# ------------------------------------------------------------
# 9. Quick diagnostics: importance ranking + gender direction
# ------------------------------------------------------------
imp = (pd.DataFrame({"feature": FEATURES, "importance": rf.feature_importances_})
       .sort_values("importance", ascending=False).reset_index(drop=True))
print(imp.to_string(index=False))
tmp = X_test.copy(); tmp["proba"] = y_proba
print("VIP prob by gender code:\n", tmp.groupby("code_gender")["proba"].mean().round(4))

# ------------------------------------------------------------
# 10. Model-based CDR across groups
# ------------------------------------------------------------
cdr_tbl = (pd.DataFrame({"group": group_test.values, "proba": y_proba})
           .groupby("group")["proba"].mean()
           .reindex(["advantaged", "disadvantaged", "other"]) * 100).round(3)
print(cdr_tbl)
print("Model-based CDR:", round(cdr_tbl["advantaged"] / cdr_tbl["disadvantaged"], 1), "x")

# ------------------------------------------------------------
# 11. Checkpoint everything Notebook 2 needs
# ------------------------------------------------------------
artifacts = {
    "rf": rf,
    "X_train": X_train, "y_train": y_train,
    "X_test": X_test,   "y_test": y_test,
    "group_test": group_test,
    "FEATURES": FEATURES, "IMMUTABLE": IMMUTABLE,
    "ACTIONABLE": ACTIONABLE, "NON_CONTROLLABLE": NON_CONTROLLABLE,
    "TARGET": TARGET,
    "cdr_table": cdr_tbl,
    "cutoffs": {"short": float(TENURE_SHORT), "long": float(TENURE_LONG)},
}
with open(os.path.join(ARTIFACT_DIR, "nb1_artifacts.pkl"), "wb") as f:
    pickle.dump(artifacts, f)
print("\nSaved nb1_artifacts.pkl ->", ARTIFACT_DIR)

Extracted: 01.카드 회원정보.zip
Extracted: 02.카드 신용정보.zip
Extracted: 03.카드 승인매출정보.zip
approval  : 6 files
credit    : 6 files
member    : 6 files
Loaded: (3000000, 9) (3000000, 7) (3000000, 2)
Merged: (3000000, 16) | duplicate member_no: 0
VIP rate: 0.1992
Cutoffs: short<=18m, long>=105m
                  size      mean
group                           
advantaged       44315  0.396186
disadvantaged    47120  0.020522
other          2908565  0.199091
Sample VIP rate: 0.1992
group
other            193905
disadvantaged      3141
advantaged         2954
Name: count, dtype: int64
F1     : 0.664
Recall : 0.5716
AUC-ROC: 0.8915
              precision    recall  f1-score   support

           0     0.9003    0.9627    0.9305     40040
           1     0.7921    0.5716    0.6640      9960

    accuracy                         0.8848     50000
   macro avg     0.8462    0.7671    0.7972     50000
weighted avg     0.8788    0.8848    0.8774     50000

         feature  importance
  amt_credit_r3m    0